# Caltech101 on Databricks Spark

Run this notebook on a Databricks cluster with GPU workers and `pyspark.ml.torch.distributor.TorchDistributor` available. Install `lit-deltalake`, `datasets`, and `torchvision` on every worker.

The preparation script writes `data/caltech101/train` and `data/caltech101/test`. These Delta tables provide binary `image` values and integer `label` values. Pass those table URIs and the process count before launching DDP.


In [ ]:
"""Train Caltech101 from Delta tables on a Databricks Spark cluster.

The Delta tables must contain ``image`` binary data and integer ``label`` values.
Prepare them with ``uv run python scripts/prep_caltech.py``. Install ``lit-deltalake``
and ``torchvision`` on the Databricks workers. For a local run, use the matching
script in ``examples/caltech``.
"""

In [ ]:
from argparse import ArgumentParser, Namespace
from dataclasses import dataclass
from typing import Any

In [ ]:
import lightning as lightning
import torch
from torch import nn
from torchvision.io import decode_image

In [ ]:
from lit_deltalake.datamodules import DeltaDataModule

from lit_deltalake.datasets import ColumnSpec, DeltaIterableDataset, DeltaScan
from lit_deltalake.readers import DeltaRsReader

In [ ]:
@dataclass(frozen=True, slots=True)
class TrainingConfig:
    train_uri: str
    test_uri: str
    batch_size: int
    epochs: int
    learning_rate: float
    num_classes: int
    num_workers: int
    version: int | None

In [ ]:
def _decode_image(value: bytes) -> Any:
    return decode_image(torch.frombuffer(bytearray(value), dtype=torch.uint8))

In [ ]:
def _image_transform() -> Any:
    from torchvision import transforms

    return transforms.Compose(
        (
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        )
    )

In [ ]:
def _dataset_factory(table_uri: str, config: TrainingConfig, shuffle: bool) -> DeltaIterableDataset:
    transform = _image_transform()
    return DeltaIterableDataset(
        table_uri,
        DeltaRsReader(),
        DeltaScan(table_uri, columns=("image", "label"), version=config.version),
        column_specs=(
            ColumnSpec("image", decoder=_decode_image, transform=transform),
            ColumnSpec("label", transform=int),
        ),
        shuffle_buffer_size=config.batch_size * 16 if shuffle else 0,
    )

In [ ]:
class CaltechDataModule(DeltaDataModule):
    """Create Delta-backed train, validation, and test DataLoaders."""

    def __init__(self, config: TrainingConfig) -> None:
        super().__init__(
            train_factory=lambda: _dataset_factory(config.train_uri, config, shuffle=True),
            validation_factory=lambda: _dataset_factory(config.test_uri, config, shuffle=False),
            test_factory=lambda: _dataset_factory(config.test_uri, config, shuffle=False),
            batch_size=config.batch_size,
            num_workers=config.num_workers,
            pin_memory=torch.cuda.is_available(),
            multiprocessing_context="spawn" if config.num_workers else None,
        )
        self.num_classes = config.num_classes

In [ ]:
class CaltechClassifier(lightning.LightningModule):
    """Fine-tune MobileNetV3 for Caltech101 classification."""

    def __init__(self, num_classes: int, learning_rate: float) -> None:
        super().__init__()
        from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large

        self.save_hyperparameters()
        self.model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
        classifier = self.model.classifier
        classifier[-1] = nn.Linear(classifier[-1].in_features, num_classes)
        self.learning_rate = learning_rate

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.model(images)

    def training_step(self, batch: dict[str, torch.Tensor], batch_index: int) -> torch.Tensor:
        del batch_index
        return self._step(batch, "train")

    def validation_step(self, batch: dict[str, torch.Tensor], batch_index: int) -> None:
        del batch_index
        self._step(batch, "val")

    def test_step(self, batch: dict[str, torch.Tensor], batch_index: int) -> None:
        del batch_index
        self._step(batch, "test")

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

    def _step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
        logits = self(batch["image"])
        labels = batch["label"].to(dtype=torch.long)
        loss = nn.functional.cross_entropy(logits, labels)
        accuracy = (logits.argmax(dim=1) == labels).float().mean()
        self.log(f"{stage}_loss", loss, on_step=stage == "train", on_epoch=True, prog_bar=stage != "train")
        self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

In [ ]:
def train_distributed(config: TrainingConfig) -> None:
    """Run inside one Spark TorchDistributor worker process."""
    torch.set_float32_matmul_precision("medium")
    data_module = CaltechDataModule(config)
    trainer = lightning.Trainer(
        accelerator="auto",
        devices="auto",
        strategy="ddp",
        max_epochs=config.epochs,
        callbacks=[
            lightning.pytorch.callbacks.EarlyStopping(monitor="val_loss", mode="min"),
            lightning.pytorch.callbacks.ModelCheckpoint(monitor="val_loss", mode="min"),
        ],
    )
    model = CaltechClassifier(data_module.num_classes, config.learning_rate)
    trainer.fit(model, datamodule=data_module)
    trainer.test(model, datamodule=data_module)

In [ ]:
def _arguments() -> ArgumentParser:
    parser = ArgumentParser(description="Train Caltech101 on a Databricks Spark cluster with DDP.")
    parser.add_argument("train_uri")
    parser.add_argument("test_uri")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--learning-rate", type=float, default=2e-4)
    parser.add_argument("--num-classes", type=int, default=101)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--version", type=int)
    parser.add_argument("--torch-distributor-processes", type=int, required=True)
    return parser

In [ ]:
def _config(arguments: Namespace) -> TrainingConfig:
    if arguments.epochs < 1:
        raise ValueError("epochs must be positive.")
    if arguments.num_classes < 2:
        raise ValueError("num_classes must be at least 2.")
    if arguments.num_workers < 0:
        raise ValueError("num_workers must not be negative.")
    if arguments.torch_distributor_processes < 1:
        raise ValueError("torch_distributor_processes must be positive.")
    return TrainingConfig(
        train_uri=arguments.train_uri,
        test_uri=arguments.test_uri,
        batch_size=arguments.batch_size,
        epochs=arguments.epochs,
        learning_rate=arguments.learning_rate,
        num_classes=arguments.num_classes,
        num_workers=arguments.num_workers,
        version=arguments.version,
    )

In [ ]:
def main() -> None:
    arguments = _arguments().parse_args()
    config = _config(arguments)
    from pyspark.ml.torch.distributor import TorchDistributor

    distributor = TorchDistributor(
        num_processes=arguments.torch_distributor_processes,
        local_mode=False,
        use_gpu=True,
    )
    distributor.run(train_distributed, config)

In [ ]:
if __name__ == "__main__":
    main()